# Full example → обучение на одной траектории

Alice и Bob решают задачу о кубе через **AsyncLLM**, затем **Qwen3-1.7B** обучается
на одной записанной траектории **Qwen3-4B-Thinking-2507**.
Промпт, начальные реплики и заголовки взяты из
[Hogwild full_example](https://github.com/eqimp/hogwild_llm/blob/main/full_example.ipynb).
Текст и token IDs начального промпта совпадают с QwQ-32B из исходного примера.
Периодические `insert_s1_prompt_every_tokens` отключены.

Запуск: kernel **AsyncLLM (uv)** → **Run All**. Основные функции находятся в ячейках ниже.
На imladris используем свободную RTX A4000 16 ГБ; FP32 веса и AdamW при обучении хранятся в RAM.

In [1]:
import asyncio
import gc
import inspect
import json
import os
import subprocess
import sys
import time
from contextlib import aclosing
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, display

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "python/minisgl/training/replay.py").is_file()
)
DEMO = ROOT / "scripts/parallel_thinking"
for path in (ROOT / "python", DEMO, ROOT / "scripts/async_thoughts"):
    sys.path.insert(0, str(path))

GPU_INDEX = 2  # Physical index from nvidia-smi; select a free 16 GiB Ampere GPU.
TEACHER = str(ROOT / "models/Qwen3-4B-Thinking-2507")
STUDENT = str(ROOT / "models/Qwen3-1.7B")
MAX_STEPS = 1024  # One token per worker per round.
TRAIN_STEPS = 80
RUN = (
    ROOT
    / "training_runs/parallel_thinking"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
RUN.mkdir(parents=True)

GPU = subprocess.check_output(
    ["nvidia-smi", f"--id={GPU_INDEX}", "--query-gpu=uuid", "--format=csv,noheader"], text=True
).strip()
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
os.environ["PATH"] = (
    f"{Path(sys.executable).parent}:{os.environ['CUDA_HOME']}/bin:" + os.environ["PATH"]
)
# TVM's default probe ignores CUDA_VISIBLE_DEVICES on this mixed-GPU machine.
os.environ["TVM_FFI_CUDA_ARCH_LIST"] = subprocess.check_output(
    ["nvidia-smi", f"--id={GPU}", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).strip()
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MAX_JOBS", "8")

import torch
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay
from minisgl.training.sft import evaluate, supervised_loss
from minisgl.training.trace import Trace
from transformers import AutoModelForCausalLM, AutoTokenizer

WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)
print(f"Teacher: {TEACHER}\nStudent: {STUDENT}\nGPU: {GPU_INDEX} ({GPU})\nRun: {RUN}")

/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Teacher: /home/uskovev/async_llm/models/Qwen3-4B-Thinking-2507
Student: /home/uskovev/async_llm/models/Qwen3-1.7B
GPU: 2 (GPU-fe2d8dfd-06f2-a5c4-a7fd-4a5f23947005)
Run: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T152010552770Z


## 1. Генерация учителя

Каждый раунд — два `one_token` через `asyncio.gather` и общий барьер.
Alice видит `[common, header, Bob, separator, Alice]`, Bob — `[common, header, Alice, separator, Bob]`.
Незавершённые блоки видны сразу; завершённые переносятся в `common`.
Это порт алгоритма на AsyncLLM, а не обещание побитового совпадения с backend Hogwild.
`DemoRecorder` записывает реальные batch, видимость блоков, append/clear и токены-цели.

In [2]:
async def together(*coroutines):
    """On an error/cancellation, settle all requests before cache cleanup."""
    tasks = [asyncio.create_task(coro) for coro in coroutines]
    try:
        return await asyncio.gather(*tasks)
    except BaseException:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        raise


async def one_token(llm, context, forbidden):
    # Close each finite stream explicitly; the pending token stays in context.
    async with aclosing(
        llm.async_generate(
            context,
            max_steps=1,
            forbid_ids=forbidden,
            sampling_params=SamplingParams(temperature=0.0, max_tokens=1),
        )
    ) as stream:
        return await anext(stream)

In [3]:
async def solve_parallel(
    model, *, max_steps=1024, memory_ratio=0.65, trace_output=None, prompt_ids=None
):
    MODEL = str(model)
    MAX_STEPS = max_steps
    MEMORY_RATIO = memory_ratio
    FINISHER_MAX_TOKENS = 64
    started = time.perf_counter()
    if torch.cuda.is_initialized():
        raise RuntimeError(
            "Restart the kernel and Run All: minisgl Engine must initialize CUDA itself."
        )
    llm = AsyncLLM(
        MODEL,
        dtype=torch.bfloat16,
        memory_ratio=MEMORY_RATIO,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    if trace_output is not None:
        from async_thoughts.training import DemoRecorder

        llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)
        # The original QwQ prompt also matches the Thinking-2507 template exactly.
        fmt = MathFormatting(tokenizer, WORKERS)
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )
        # Graph profiles admit two requests, so initialize the long prompt first.
        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]
        # Leave the last prefix token pending, so round 1 is a joint decode.
        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>", "</think>")
            if t in tokenizer.vocab
        ]
        # EOS remains a stop condition, even if a model uses endoftext as EOS.
        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        for iteration in range(1, MAX_STEPS + 1):
            new_tokens = await together(*(one_token(llm, ctx, forbidden) for ctx in contexts))
            for i, token in enumerate(new_tokens):
                current[i].append(token)
                generated[i] += 1

            # Both futures are settled: mutations below cannot race a decode.
            finished = [i for i in range(2) if fmt.is_end_of_step(current[i])]
            await together(
                *(llm.forward(cache_view=contexts[i], return_logits=False) for i in finished)
            )
            for i in finished:
                block = worker_blocks[i]
                assert block.token_ids == current[i], "A boundary token was lost or duplicated."
                await llm.append_block(common, block)
                history.extend(current[i])
                assert common.token_ids == initial_ids + history
                events.append(
                    {
                        "round": iteration,
                        "worker": WORKERS[i],
                        "step": step_indices[i],
                        "tokens": len(current[i]),
                    }
                )
                await llm.free_block(block)  # Reset this same block; views keep their references.
                assert block.num_tokens == 0
                step_indices[i] += 1
                start = fmt.get_step_prefix(WORKERS[i], step_indices[i])
                current[i] = encode(start)

            # Prefill prefix[:-1] after all merges; keep the last token pending.
            await together(
                *(
                    llm.forward(current[i][:-1], cache_view=contexts[i], return_logits=False)
                    for i in finished
                )
            )
            for i in finished:
                contexts[i].next_input_id = current[i][-1]

            for i, ctx in enumerate(contexts):
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]
            if iteration % 128 == 0:
                print(f"round {iteration}: {len(events)} completed steps", flush=True)
            if eos in new_tokens:
                break

        # Flush both final sampled tokens before the answer continuation reads them.
        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=FINISHER_MAX_TOKENS,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=FINISHER_MAX_TOKENS),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": MODEL,
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        if trace_output is not None:
            llm.save(
                trace_output, model=MODEL, problem=PROBLEM, thinker_tokens=[], writer_tokens=[]
            )
        return result
    finally:
        # No active streams or queued requests remain here, including on errors.
        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

Для освобождения GPU между этапами запускаем функции в отдельных процессах.
`run_stage` берёт исходный код **уже выполненных ячеек** через `inspect.getsource`;
сохраняет его и лог в каталоге запуска. Отдельного модуля с реализацией нет.

In [4]:
def run_stage(name, function, *args, **kwargs):
    free = int(
        subprocess.check_output(
            [
                "nvidia-smi",
                f"--id={GPU}",
                "--query-gpu=memory.free",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        ).strip()
    )
    if free < 14000:
        raise RuntimeError(f"Only {free} MiB free on GPU {GPU_INDEX}; select a free GPU")
    # Capture current cell definitions, including edits not yet saved to the .ipynb.
    definitions = [together, one_token, solve_parallel]
    if function is train:
        definitions += [load_training_example, evaluate_student, train]
    imports = """
import asyncio, gc, json, sys, time
from pathlib import Path
from contextlib import aclosing
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay
from minisgl.training.trace import Trace
from minisgl.training.sft import evaluate, supervised_loss
from async_thoughts.training import DemoRecorder
"""
    result_path = RUN / f"{name}.json"
    preamble = f"import sys\nsys.path[:0] = {[str(ROOT / 'python'), str(DEMO), str(ROOT / 'scripts/async_thoughts')]!r}\n"
    constants = f"\nWORKERS = {WORKERS!r}\nPROBLEM = {PROBLEM!r}\n"
    invocation = f"{function.__name__}(*request['args'], **request['kwargs'])"
    if inspect.iscoroutinefunction(function):
        invocation = f"asyncio.run({invocation})"
    worker = preamble + imports + constants + "\n\n".join(inspect.getsource(f) for f in definitions)
    worker += f"\nrequest = json.load(sys.stdin)\nresult = {invocation}\n"
    worker += f"Path({str(result_path)!r}).write_text(json.dumps(result, indent=2))\n"
    script = RUN / f"{name}.py"
    script.write_text(worker)
    log_path = RUN / f"{name}.log"
    print(f"Running {name}; log: {log_path}", flush=True)
    with log_path.open("w") as log:
        subprocess.run(
            [sys.executable, "-u", str(script)],
            input=json.dumps({"args": args, "kwargs": kwargs}, default=str),
            text=True,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
            cwd=ROOT,
        )
    return json.loads(result_path.read_text())

## 2. Обучение маленькой модели

`Replay` восстанавливает видимость блоков и позиции RoPE из trace. Loss считается
только по сгенерированным токенам; prefill и flush имеют target `-100`.
`chunk_size=64` ограничивает временную память и не обрезает контекст.
`evaluate` импортирован из `python/minisgl/training/sft.py`: это teacher-forced проверка.

BF16 forward/backward выполняется на GPU; FP32 master weights, градиенты и AdamW — на CPU.
Частоты RoPE остаются FP32. Обучаем все веса на одной траектории, максимум 80 обновлений.

In [5]:
def load_training_example(trace_path, model_path):
    data = json.loads(Path(trace_path).read_text())
    (example,) = data["examples"]
    trace = Trace.from_dict(example["trace"])
    targets = torch.tensor(example["targets"], dtype=torch.long)
    assert len(targets) == trace.num_tokens and targets.ne(-100).any()
    teacher = AutoTokenizer.from_pretrained(data["model"])
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    teacher_vocab = {i: t for t, i in teacher.get_vocab().items()}
    student_vocab = {i: t for t, i in tokenizer.get_vocab().items()}
    used = {t.token_id for t in trace.tokens} | set(targets[targets.ne(-100)].tolist())
    assert all(teacher_vocab[i] == student_vocab.get(i) for i in used), "Token IDs differ"
    replay = Replay.from_trace(
        trace, query_chunk_size=64, attention_checkpointing=True, cache_plans=True
    )
    return replay, targets, tokenizer


def evaluate_student(model, replay, targets):
    result = evaluate(model.eval(), [(replay, targets)], chunk_size=64)
    return {k: result[k] for k in ("loss", "accuracy")}

In [6]:
def train(trace_path, model_path, output, *, steps=80, lr=3e-5):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    torch.manual_seed(7)
    torch.backends.cuda.matmul.allow_tf32 = False
    replay, targets, tokenizer = load_training_example(trace_path, model_path)

    student = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32)
    # CPU FP32 masters keep sub-BF16 updates; only BF16 compute weights live on GPU.
    masters = {
        name: torch.nn.Parameter(p.detach().clone()) for name, p in student.named_parameters()
    }
    del student
    # Load compute weights through HF so RoPE buffers retain their FP32 dtype.
    # model.to(dtype=BF16) would also round inv_freq and change replay positions.
    student = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16).cuda()
    assert student.config.model_type == "qwen3", "Only dense Qwen3 is supported"
    assert student.model.rotary_emb.inv_freq.dtype == torch.float32, (
        "RoPE frequencies must stay FP32"
    )
    student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    optimizer = torch.optim.AdamW(masters.values(), lr=lr, foreach=False)
    initial = final = evaluate_student(student, replay, targets)
    history = [{"step": 0, **initial}]
    print(f"initial: {initial}", flush=True)
    started = time.perf_counter()
    step = 0
    for step in range(1, steps + 1):
        student.train()
        optimizer.zero_grad(set_to_none=True)
        student.zero_grad(set_to_none=True)
        loss = supervised_loss(student, replay, targets, chunk_size=64) / targets.ne(-100).sum()
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite loss")
        loss.backward()
        for name, parameter in student.named_parameters():
            assert parameter.grad is not None, f"Missing gradient: {name}"
            masters[name].grad = parameter.grad.detach().to(device="cpu", dtype=torch.float32)
            parameter.grad = None
        norm = torch.nn.utils.clip_grad_norm_(masters.values(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        with torch.no_grad():
            for name, parameter in student.named_parameters():
                parameter.copy_(masters[name])
        print(
            f"update {step}: loss={float(loss.detach()):.6f}; grad_norm={float(norm):.4f}",
            flush=True,
        )
        del loss
        if step % 5 == 0 or step == steps:
            final = evaluate_student(student, replay, targets)
            history.append({"step": step, **final})
            (output / "progress.json").write_text(json.dumps(history, indent=2))
            print(f"evaluation {step}: {final}", flush=True)
            if final["accuracy"] == 1.0 and final["loss"] < 0.01:
                break
    checkpoint = output / "checkpoint"
    # Save the full-precision master values, not their rounded BF16 compute copies.
    student = student.cpu().float()
    with torch.no_grad():
        for name, parameter in student.named_parameters():
            parameter.copy_(masters[name])
    student.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    metrics = {
        "initial": initial,
        "final": final,
        "history": history,
        "steps": step,
        "supervised_tokens": int(targets.ne(-100).sum()),
        "training_seconds": time.perf_counter() - started,
        "peak_cuda_gib": torch.cuda.max_memory_allocated() / 2**30,
        "checkpoint": str(checkpoint),
    }
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2))
    assert final["loss"] < initial["loss"], "Training did not decrease loss"
    return metrics

## Запуск: одна траектория → один student

Сначала выполняем исходные 1024 раунда учителя и сохраняем trace. Для краткой проверки
численного ответа используется завершающая реплика `MathFormatting.s1_finisher_suffix`.
Ответ 294 проверяется после генерации и не передаётся модели в промпте.

In [7]:
TRACE = RUN / "teacher_trace.json"
teacher = run_stage("teacher", solve_parallel, TEACHER, max_steps=MAX_STEPS, trace_output=TRACE)
assert teacher["answer"] == 294, "Teacher answer is wrong; inspect the trajectory before training"
display(Markdown(teacher["transcript"] + "\n\n" + teacher["answer_text"]))
print("Published blocks:", teacher["completed_steps"])

Running teacher; log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T152010552770Z/teacher.log


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll compute the distances between the points to find the edge length of the cube. Let's start with PQ, QR, and RP.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three vertices. First, I should find the distances between P, Q, and R. Let's compute PQ, QR, and RP.

**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)
**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If the triangle is formed by three vertices where each pair is connected by a face diagonal, then the side length of the triangle would be s*sqrt(2). But if it's formed by a space diagonal, it would be s*sqrt(3). Wait, but here the side length is 7*sqrt(2), so if it's a face diagonal, then s*sqrt(2) = 7*sqrt(2) => s = 7. If it's a space diagonal, s*sqrt(3) = 7*sqrt(2) => s = 7*sqrt(2/3), which is not an integer, but the problem says it's a cube, so probably s is integer. So likely it's a face diagonal.



### Work in progress (others)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If<...>

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR =<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}

Published blocks: 6


In [8]:
metrics = run_stage("train", train, TRACE, STUDENT, RUN / "training", steps=TRAIN_STEPS)
print(f"Loss: {metrics['initial']['loss']:.6f} → {metrics['final']['loss']:.6f}")
print(f"Token accuracy: {metrics['initial']['accuracy']:.2%} → {metrics['final']['accuracy']:.2%}")
print(f"Updates: {metrics['steps']}; supervised tokens: {metrics['supervised_tokens']}")
assert metrics["final"]["accuracy"] == 1.0 and metrics["final"]["loss"] < 0.01

Running train; log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T152010552770Z/train.log


Loss: 0.169990 → 0.000211
Token accuracy: 95.03% → 100.00%
Updates: 15; supervised tokens: 2052


## Свободная генерация из сохранённого checkpoint

Student получает только тот же начальный промпт. Токены учителя и события публикации
ему не подаются. Сравниваем token IDs и порядок публикаций; совпадение относится
к этому алгоритму с барьером на каждом раунде, а не к времени исполнения.

In [9]:
student = run_stage(
    "student",
    solve_parallel,
    metrics["checkpoint"],
    max_steps=MAX_STEPS,
    prompt_ids=teacher["prompt_ids"],
)
comparison = {
    "teacher_answer": teacher["answer"],
    "student_answer": student["answer"],
    "same_token_ids": all(
        student[k] == teacher[k] for k in ("prompt_ids", "history_ids", "current_ids", "answer_ids")
    ),
    "same_publications": student["merge_events"] == teacher["merge_events"],
}
print(json.dumps(comparison, indent=2))
(RUN / "comparison.json").write_text(json.dumps(comparison, indent=2))
display(Markdown(student["answer_text"]))
assert student["answer"] == 294, "Student free generation did not solve the problem"

Running student; log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T152010552770Z/student.log


{
  "teacher_answer": 294.0,
  "student_answer": 294.0,
  "same_token_ids": true,
  "same_publications": true
}




Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}